# Pose completion: PoseD-Flow's protocol (their Tab. 2), PoseD-Flow and RSGM

The rotations of a body part are hidden, and a model fills them in: 10 hypotheses per pose. Everything follows PoseD-Flow (Nadar, Foti, Birdal, CVPR 2026; their `pose_completion.py`), `hp/evaluation/completion.py`:

- **Data**: their AMASS test set (SMPL fits, DPoser's preprocessing), every 10th pose, 5000 poses (their batches of 500, the last one dropped).
- **Occlusions**: left leg, legs, arms, trunk (their `BodyPartIndices`; "trunk" is the three spine joints and the two shoulders).
- **Metrics** on SMPL-X neutral, betas = 0 (their body model): MPVPE over the vertices of the hidden part, MPJPE over the hidden joints, mean, std and min over the 10 hypotheses; APD of the hidden joints over the hypotheses (cm). Their table reports MPVPE mean ± std and APD. Added here: min over the hypotheses (best of 10), MPJPE, MGEO of the hidden joints (rad).
- **Method**: every model is completed by **D-Flow** (`hp/models/dflow.py`) with their settings: the start of the ODE is optimized so that the visible joints of the result match (geodesic loss), 200 Adam steps (300 for the arms), lr 0.1, 100 Euler steps of the ODE, running cost 1e-3, and the start mixes noise with the sampling ODE run from the observed pose. Only the ODE inside differs: PoseD-Flow's field, or RSGM's probability flow. Their RiemannianAdam is Adam on the tangent coordinates here.

**Joints.** RSGM has 19 joints (no toes): a hidden toe is set to rest. The columns *toes at rest* set the toes to rest in the ground truth and in every completion, which compares 19- and 21-joint models alike.

**The check of the protocol**: PoseD-Flow's checkpoint should give about their PoseRFM row (left leg 83.81 ± 29.07, APD 6.02). Differences come from the optimizer (Adam vs RiemannianAdam) and the random draws.

**Computing.** D-Flow optimizes every hypothesis (about 0.2–0.3 s per step for 500 poses on mps: several hours per model). Run `scripts/evaluate_completion.py` (e.g. on a server); this notebook reads its cache and computes only if `COMPUTE_DEVICE` is set.

In [ ]:
import sys
from pathlib import Path

SRC = next(str(p / "src") for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "hp").is_dir())
if SRC not in sys.path:
    sys.path.insert(0, SRC)

from hp.evaluation.completion import OCCLUSIONS, PAPER, Body, cached_metrics, cached_run, summary, test_poses

# ---- parameters ----------------------------------------------------------------------
ROTATIONS = "~/data/AMASS_rotations_posedflow_c10"  # as scripts/evaluate_completion.py
CACHE = "~/data/experiments/completion/cache"
MODELS = {  # as scripts/evaluate_completion.py
    "PoseD-Flow: their checkpoint": dict(posedflow="~/data/posedflow_ckpt/checkpoints/epoch-009_step-0_loss-0.0000.ckpt"),
    "RSGM 512x4: their c10": dict(run="~/data/experiments/rsgm_posedflow_small_s10/run_0001"),
    "RSGM 1536x6: their c10": dict(run="~/data/experiments/rsgm_posedflow_s10/run_0001"),
    "RSGM 512x4: their c10, seed 11": dict(run="~/data/experiments/rsgm_posedflow_small_s11/run_0001"),
    "RSGM 512x4: their c10, seed 12": dict(run="~/data/experiments/rsgm_posedflow_small_s12/run_0001"),
    "RSGM 512x4: their full": dict(run="~/data/experiments/full/rsgm_posedflow_small_s10/run_0001"),
    # the same network with other ODEs inside D-Flow (the samplers of the Pareto comparison): half the network calls
    "RSGM 512x4: their c10, ODE Euler 50": dict(run="~/data/experiments/rsgm_posedflow_small_s10/run_0001",
                                                 prior=dict(ode_steps=50)),
    "RSGM 512x4: their c10, Heun EDM 50 NFE": dict(run="~/data/experiments/rsgm_posedflow_small_s10/run_0001",
                                                   prior=dict(sampler="heun", schedule="edm", ode_steps=25)),
    # DPoser (Euclidean diffusion on axis-angle) through the same D-Flow: separates the prior from their own completion
    "DPoser: their full": dict(run="~/data/experiments/full/dposer_posedflow_s10/run_0001"),
    "DPoser: their c10, RSGM budget": dict(run="~/data/experiments/dposer_posedflow_rsgm_budget_s10/run_0001"),
    # their start pushes the observation forward through the ODE; in R^n that leaves the typical set of N(0, I)
    "DPoser: their full, inverted start": dict(run="~/data/experiments/full/dposer_posedflow_s10/run_0001",
                                               prior=dict(invert=True)),
    "RSGM 512x4: their c10, inverted start": dict(run="~/data/experiments/rsgm_posedflow_small_s10/run_0001",
                                                  prior=dict(invert=True)),
}
POSES = None  # None: their 5000 test poses; a number: the first n (a separate cache entry)
COMPUTE_DEVICE = None  # None: only read the cache; "mps" / "cuda:0": compute what is missing
# ----------------------------------------------------------------------------------------

## 1. The completions and their metrics

In [ ]:
gt = test_poses(ROTATIONS, POSES)
body = Body()
results, rest = {}, {}
for name, spec in MODELS.items():
    for occlusion in OCCLUSIONS:
        pred = cached_run(Path(CACHE), name, spec, gt, occlusion, COMPUTE_DEVICE, rotations=ROTATIONS)
        if pred is None:
            print(f"{name} / {occlusion}: not computed yet")
            continue
        plain, toes_at_rest = cached_metrics(Path(CACHE), name, occlusion, pred, gt, body)  # computed once
        results[name, occlusion], rest[name, occlusion] = summary(plain), summary(toes_at_rest)
print(f"{len(gt)} test poses, {len(results)} of {len(MODELS) * len(OCCLUSIONS)} (model, occlusion) done")

## 2. Next to their Tab. 2

Per occlusion: their rows (MPVPE mean ± std, APD), then ours with the same columns and the added ones. The std is over the 10 hypotheses of a pose (the spread of the completions), averaged over the poses, as theirs; it is not an error bar.

In [ ]:
head = f"{'':34s} {'MPVPE':>15s} {'APD':>6s} {'best of 10':>11s} {'MPJPE':>8s} {'MGEO':>7s} {'toes at rest':>13s}"
for occlusion in OCCLUSIONS:
    print(f"\n{occlusion}\n{head}")
    for name, rows in PAPER.items():
        m, s, a = rows[occlusion]
        print(f"{name + ' (paper)':34s} {f'{m:.2f} ± {s:.2f}':>15s} {a:6.2f}")
    for name in MODELS:
        r = results.get((name, occlusion))
        if r is None:
            continue
        t = rest[name, occlusion]
        mpvpe = f"{r['mpvpe_mean']:.2f} ± {r['mpvpe_std']:.2f}"
        print(f"{name:34s} {mpvpe:>15s} {r['apd']:6.2f} {r['mpvpe_min']:11.2f} "
              f"{r['mpjpe_mean']:8.2f} {r['mgeo_mean']:7.3f} {t['mpvpe_mean']:13.2f}")
print("\nMPVPE, best of 10, MPJPE in mm (hidden part); APD in cm; MGEO in rad. Ours: D-Flow with their settings.")

## 3. Reading

- **The check**: PoseD-Flow's checkpoint against their PoseRFM row. A gap of a few mm is the optimizer and the draws; a large gap means the protocol differs from theirs.
- **RSGM against PoseD-Flow** is the same guidance with another prior. RSGM was trained on their c10 (1.86M poses), their checkpoint on all 18.6M.
- **MPVPE assumes one right answer** (their own remark): a plausible completion other than the ground truth counts as an error. Best of 10 and APD show the other side: whether the right pose is among the hypotheses, and how different they are.
- **Toes at rest** is the fair column between 19 and 21 joints; for the trunk and the arms it is the same as MPVPE (the toes are not hidden).